In [ ]:
#!pip install -r requirements.txt

Colab setup

In [ ]:
!git clone https://github.com

In [ ]:
cd iCSC-exercise

In [ ]:
!pip install -r requirements.txt

# Self-Attention as a stack of Finite Dimensional Ising Models: A statistical - Physics view with GPU - Accelerated Prototype

## Introduction:

The exercise is divided into three parts: 
1) Ising model 
2) Attention function
3) Ising mapped into attention function 


Ising model brute force ( You may skip )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


L = 20              # lattice size (L x L)
n_steps = 5000         # Monte Carlo steps per temperature
T_vals = np.linspace(1.5, 4.0, 25)  # temperature range
Tc =              # known critical temperature for 2D Ising

# temperatures to check at
T_low = 
T_high = 


def initialize_lattice(L):
    return np.random.choice([-1, 1], size=(L, L))

def energy(lattice):
    E = 0
    for i in range(L):
        for j in range(L):
            S = lattice[i, j]
            neighbors = (
                lattice[(i+1)%L, j] +
                lattice[(i-1)%L, j] +
                lattice[i, (j+1)%L] +
                lattice[i, (j-1)%L]
            )
            E += -S * neighbors
    return E / 2

def magnetization(lattice):
    return np.abs(np.sum(lattice)) / (L*L)


def metropolis_step(lattice, T):
    i = np.random.randint(0, L)
    j = np.random.randint(0, L)

    S = lattice[i, j]
    neighbors = (
        lattice[(i+1)%L, j] +
        lattice[(i-1)%L, j] +
        lattice[i, (j+1)%L] +
        lattice[i, (j-1)%L]
    )

    dE = 2 * S * neighbors

    if dE < 0 or np.random.rand() < np.exp(-dE / T):
        lattice[i, j] *= -1

M_vs_T = []

lattice_low = None
lattice_high = None
lattice = initialize_lattice(L)

for T in T_vals:
    for _ in range(n_steps):
        metropolis_step(lattice, T)
    
    Ms = []
    for _ in range(5000):
        metropolis_step(lattice, T)
        Ms.append(magnetization(lattice))

    M = np.mean(Ms)
    M_vs_T.append(M)
  

    if abs(T - T_low) < 0.05:
        lattice_low = lattice.copy()
    if abs(T - T_high) < 0.05:
        lattice_high = lattice.copy()


M_vs_T = np.array(M_vs_T)


fig, axes = plt.subplots(1, 3, figsize=(12, 4))

# LEFT
#axes[0].imshow(, cmap="coolwarm")
axes[0].set_title()
axes[0].axis("off")

# MIDDLE
#axes[1].plot()
axes[1].axvline(Tc, linestyle="--", label="Tc")
#axes[1].set_xlabel("")
#axes[1].set_ylabel("")
axes[1].set_title("Phase Transition")
axes[1].legend()

# RIGHT
#axes[2].imshow(, cmap="coolwarm")
axes[2].set_title()
axes[2].axis("off")

plt.tight_layout()
plt.show()

Ising model as attention activation

In [ ]:
import matplotlib.gridspec as gridspec
import seaborn as sns

np.random.seed(42)
print('You are loaded with libraries needed')
      

define model base

In [ ]:
tokens  = # make your own token list
seq_len = 
d_model = 

# alloted embeddings for a three listed token example
# there is something you will have to consider when setting the length of element in array
X = np.array([
    [1.0, 0.0, 1.0],  
    [0.0, 2.0, 0.0],  
    [1.0, 1.0, 1.0],
    #
    #   
], dtype=np.float64)

print('Token embeddings X:')
for tok, vec in zip(tokens, X):
    print(f'  {tok:6s}: {vec}')

Define Interaction energy
We model token embeddings as **spin vectors**. The interaction energy between token $i$ and token $j$ is:

$$E(i,j) = -J_{ij} = -\frac{x_i \cdot x_j}{\|x_i\| \|x_j\|}$$


Question: Can you state the assumption made here?

Answer: 

In [ ]:
def ising_energy_matrix(X):
    
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    #/(norms + 1e-8)          # shape: (seq_len, d_model)
    
    # similarity matrix: 
    cosine_sim =              # shape: (seq_len, seq_len)    
    E = -cosine_sim
    return E


E = ising_energy_matrix(X)


print('Ising energy matrix E(i,j)')
print(np.round(E, 4))
print('Interpretation: ')

Let's find where attention is being given 

In [ ]:
# Energy landscape
sns.heatmap(E, annot=True, fmt='.3f', cmap='RdYlGn_r',
            xticklabels=tokens, yticklabels=tokens,
            linewidths=0.5, ax=axes[1])
axes[1].set_title('Ising Energy E(i,j) = -J\nGreen = low energy = will attend more', fontsize=11)
axes[1].set_xlabel('Token j'); axes[1].set_ylabel('Token i')
plt.show()

In [ ]:
def boltzmann_weights(E, beta):
    """
    Compute unnormalised Boltzmann factors: w(i,j) = exp(-beta * E(i,j))
    """
    # Subtract row-max for numerical stability (same trick as log-sum-exp)
    # This doesn't change the normalised result, just prevents overflow.
    E_stable = E - E.min(axis=1, keepdims=True)
    W = np.exp(-beta * E_stable)
    return W


beta = 5.0   # inverse temperature — try changing this!

W_unnorm = boltzmann_weights(E, beta)

def ising_attention(X, beta):
    """
    Full Ising-model attention:
      1. Compute pairwise energy E(i,j) via spin alignment (cosine)
      2. Boltzmann factor: exp(-beta * E)
      3. Normalise by partition function Z (= softmax equivalent)
      4. Weighted sum of 'values' (raw embeddings X here)
    """
    E, cosine_sim = ising_energy_matrix(X)
    
    # Unnormalised Boltzmann weights (numerically stable)
    neg_bE = -beta * E
    neg_bE_stable = neg_bE - neg_bE.max(axis=1, keepdims=True)  # subtract row max
    W = np.exp(neg_bE_stable)
    
    # Partition function (denominator)
    Z = W.sum(axis=1, keepdims=True)
    
    # Normalised attention weights (Boltzmann distribution)
    A = W / Z   # shape: (seq_len, seq_len), rows sum to 1
    
    # Context output: weighted sum of value vectors (using X directly as V)
    output = A @ X
    
    return A, output, E, Z


beta = 5.0
A_ising, output_ising, E_ising, Z_ising = ising_attention(X, beta=beta)

print(f'β = {beta}')
print('\nPartition function Z_i (normalisation constant per token):')
print(np.round(Z_ising.flatten(), 4))
print('\nIsing Attention Weights A(i,j) = exp(-βE) / Z:')
print(np.round(A_ising, 4))
print('\nRow sums (should be 1.0):', np.round(A_ising.sum(axis=1), 6))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(A_ising, annot=True, fmt='.3f', cmap='Reds',
            xticklabels=tokens, yticklabels=tokens,
            linewidths=0.5, ax=ax, vmin=0, vmax=1)
ax.set_title(f'Ising Attention Weights  (β={beta})\n'
             'A(i,j) = exp(-β·E(i,j)) / Z', fontsize=12)
ax.set_xlabel('Token j  (attended to)')
ax.set_ylabel('Token i  (attending)')
plt.tight_layout()
plt.show()